In [ ]:
%pip install imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import sklearn

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import imblearn

# Adding XGBoost explicitly as required by your project spec
import xgboost as xgb

# Epic 2 - Story 1: Import and read the dataset
df = pd.read_csv("Dataset/loan_prediction.csv")

# Display the first few rows to ensure success
df

In [ ]:
# ==========================================
# EPIC 2 - STORY 2: UNIVARIATE ANALYSIS
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# Set up a clean layout for categorical plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Target Variable: Loan Status
sns.countplot(x='Loan_Status', data=df, ax=axes[0,0], palette='pastel')
axes[0,0].set_title('Distribution of Loan Approval Status (Target Variable)')

# 2. Gender Distribution
sns.countplot(x='Gender', data=df, ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Distribution of Applicant Gender')

# 3. Marital Status Distribution
sns.countplot(x='Married', data=df, ax=axes[1,0], palette='deep')
axes[1,0].set_title('Distribution of Marital Status')

# 4. Credit History Distribution
sns.countplot(x='Credit_History', data=df, ax=axes[1,1], palette='coolwarm')
axes[1,1].set_title('Distribution of Credit History (1.0 = Good, 0.0 = Bad)')

plt.tight_layout()
plt.show()

# --- Next, let's look at a Numerical Column (Income Distribution) ---
plt.figure(figsize=(10, 4))
sns.histplot(df['ApplicantIncome'], kde=True, bins=40, color='darkblue')
plt.title('Distribution of Applicant Income')
plt.xlabel('Income Amount')
plt.show()

In [ ]:
# ==========================================
# EPIC 2 - STORY 3: BIVARIATE ANALYSIS
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Credit History vs Loan Status (Categorical vs Categorical)
sns.countplot(x='Credit_History', hue='Loan_Status', data=df, ax=axes[0], palette='coolwarm')
axes[0].set_title('Loan Status by Credit History')
axes[0].set_xlabel('Credit History (1.0 = Good, 0.0 = Bad)')

# 2. Married Status vs Loan Status (Categorical vs Categorical)
sns.countplot(x='Married', hue='Loan_Status', data=df, ax=axes[1], palette='pastel')
axes[1].set_title('Loan Status by Marital Status')

plt.tight_layout()
plt.show()

# 3. Applicant Income vs Loan Status (Numerical vs Categorical)
plt.figure(figsize=(10, 5))
sns.boxplot(x='Loan_Status', y='ApplicantIncome', data=df, palette='Set2')
plt.title('Applicant Income Distribution by Loan Status')
plt.ylim(0, 30000) # Capping y-axis temporarily to see the boxes clearly despite outliers
plt.show()

In [ ]:
# ==========================================
# EPIC 2 - STORY 4: MULTIVARIATE ANALYSIS
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(10, 8))

# Select only the numeric columns for correlation matrix
numeric_df = df.select_dtypes(include=[np.number])

# Calculate the Pearson correlation matrix
corr_matrix = numeric_df.corr()

# Plot the heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Multivariate Correlation Heatmap of Numeric Features')
plt.show()

In [ ]:
# ==========================================
# EPIC 3 - STORY 1: CHECKING MISSING VALUES
# ==========================================

# Count missing values in each column
missing_values = df.isnull().sum()

# Display only columns that have missing values
print("--- Missing Values Count per Column ---")
print(missing_values[missing_values > 0])

In [ ]:
# Handling Categorical Columns with Mode
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

# Handling Numerical Columns with Median
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())

# Verify that all missing values are completely gone
print("Remaining missing values:", df.isnull().sum().sum())

In [ ]:
# # ==========================================
# # EPIC 3 - STORY 2: BALANCING THE DATASET
# # ==========================================
# from sklearn.preprocessing import LabelEncoder
# from imblearn.over_sampling import SMOTE

# # Drop Loan_ID since it's an arbitrary tracking string, not a pattern
# if 'Loan_ID' in df.columns:
#     df = df.drop(columns=['Loan_ID'])

# # Initialize LabelEncoder
# le = LabelEncoder()

# # List of text columns that need converting to numbers
# categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

# # Convert columns one by one
# for col in categorical_cols:
#     df[col] = le.fit_transform(df[col].astype(str))

# print("Categorical columns encoded successfully! Previewing numeric structure:")
# df.head()

# ==========================================
# EPIC 3 - STORY 2: BALANCING THE DATASET
# ==========================================
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# Drop Loan_ID since it's an arbitrary tracking string, not a pattern
if 'Loan_ID' in df.columns:
    df = df.drop(columns=['Loan_ID'])

# List of text columns that need converting to numbers
categorical_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Education',
    'Self_Employed',
    'Property_Area',
    'Loan_Status'
]

# Create a dictionary to store one encoder per column
label_encoders = {}

# Encode each categorical column and save its encoder
for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))
    label_encoders[col] = encoder

print("Categorical columns encoded successfully!")
df.head()

In [ ]:
# Separate features (X) from the target outcome variable (y)
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

print(f"Original class distribution:\n{y.value_counts()}")

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Resample the features and target
X_balanced, y_balanced = smote.fit_resample(X, y)

print(f"\nBalanced class distribution:\n{y_balanced.value_counts()}")

In [ ]:
# ==========================================
# EPIC 3 - STORIES 3 & 4: SCALING AND SPLITTING
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1: Split the balanced dataset into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

# Step 2: Initialize the StandardScaler
scaler = StandardScaler()

# Step 3: Fit and transform the Training features, and transform the Testing features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verify the final data dimensions
print("--- Data Preprocessing Complete ---")
print(f"Training features shape : {X_train_scaled.shape}")
print(f"Testing features shape  : {X_test_scaled.shape}")
print(f"Training labels count   : {y_train.value_counts().to_dict()}")
print(f"Testing labels count    : {y_test.value_counts().to_dict()}")

In [ ]:
import joblib

# Bundle and save all preprocessed data variables
# preprocessing_bundle = {
#     'X_train': X_train_scaled,
#     'X_test': X_test_scaled,
#     'y_train': y_train,
#     'y_test': y_test,
#     'scaler': scaler
# }

preprocessing_bundle = {
    'X_train': X_train_scaled,
    'X_test': X_test_scaled,
    'y_train': y_train,
    'y_test': y_test,
    'scaler': scaler,
    'label_encoders': label_encoders
}

joblib.dump(preprocessing_bundle, 'models/preprocessed_loan_data.pkl')
print("Success! Preprocessed data saved as 'preprocessed_loan_data.pkl'")

In [ ]:
import joblib

data = joblib.load("models/preprocessed_loan_data.pkl")

print(data.keys())

In [ ]:
encoders = data["label_encoders"]

for column, encoder in encoders.items():
    print(column)
    print(encoder.classes_)
    print()